In [10]:
import pandas as pd 
import numpy as np

In [ ]:
df1 = pd.read_excel('Datos_Prueba.xlsx', 'Libro_Auxiliar')
df2 = pd.read_excel('Datos_Prueba.xlsx', 'Extracto_Bancario')

# Marcamos los duplicados ANTES del merge, en cada hoja por separado
# keep=False marca TODAS las filas involucradas en un duplicado
df1["Duplicado_Libro"] = np.where(df1["Referencia"].duplicated(keep=False), "SI", "")
df2["Duplicado_Banco"] = np.where(df2["Referencia"].duplicated(keep=False), "SI", "")

In [ ]:
# Deteccion de referencias sospechosas: misma referencia normalizada pero texto original distinto
def normalizar(ref):
    return str(ref).strip().upper().replace(" ", "").replace("-", "")

tmp1 = df1[["Referencia"]].assign(Ref_Norm=df1["Referencia"].map(normalizar))
tmp2 = df2[["Referencia"]].assign(Ref_Norm=df2["Referencia"].map(normalizar))

sospechosas = tmp1.merge(tmp2, on="Ref_Norm", suffixes=("_libro", "_banco"))
sospechosas = sospechosas[
    sospechosas["Referencia_libro"] != sospechosas["Referencia_banco"]
].drop_duplicates()

# Referencias sin contrapartida ni siquiera normalizada (partidas legitimas o errores)
sin_libro = df1[~df1["Referencia"].map(normalizar).isin(df2["Referencia"].map(normalizar))]
sin_banco = df2[~df2["Referencia"].map(normalizar).isin(df1["Referencia"].map(normalizar))]

print(f"Pares sospechosos detectados: {len(sospechosas)}")
sospechosas

In [12]:
df = pd.merge(df1, df2, on="Referencia", how="outer")

In [13]:
df["Coinciliado"] = np.where(df["Monto_x"] == df["Monto_y"], "SI", "NO")

In [14]:
en_libro_sin_banco = df["Monto_x"].notna() & df["Monto_y"].isna()   # solo en el libro auxiliar
en_banco_sin_libro = df["Monto_x"].isna() & df["Monto_y"].notna()   # solo en el extracto

df["Partida en Tránsito / Nota no Registrada"] = np.select(
    [en_libro_sin_banco, en_banco_sin_libro],
    ["Partida en Tránsito", "Nota no Registrada"],
    default=""
)

In [15]:
df["Diferencia de Montos"] = np.where(df["Monto_x"] - df["Monto_y"] != 0, df["Monto_x"] - df["Monto_y"],np.nan)

In [16]:
df

,Fecha_x,Referencia,Concepto,Monto_x,Fecha_y,Descripcion_Banco,Monto_y,Coinciliado,Partida en Tránsito / Nota no Registrada,Diferencia de Montos
0,2026-10-02,CHQ-001,Pago proveedor X,-1200.0,2026-10-04,Cobro cheque taquilla,-1200.0,SI,,NaN
1,2026-10-05,CHQ-002,Pago arriendo,-2000.0,NaT,NaN,NaN,NO,Partida en Tránsito,NaN
2,2026-10-29,CHQ-003,Pago proveedor W,-1900.0,NaT,NaN,NaN,NO,Partida en Tránsito,NaN
3,2026-10-21,CHQ-005,Pago proveedor Z,-800.0,2026-10-21,Cobro cheque taquilla,-800.0,SI,,NaN
4,2026-10-22,CHQ-005,Pago proveedor Z,-800.0,2026-10-21,Cobro cheque taquilla,-800.0,SI,,NaN
5,NaT,DEV-001,NaN,NaN,2026-10-17,Nota debito cheque devuelto,-6200.0,NO,Nota no Registrada,NaN
6,NaT,DOM-001,NaN,NaN,2026-10-27,Debito automatico seguro,-350.0,NO,Nota no Registrada,NaN
7,2026-10-01,FAC-1001,Cobro cliente A,5000.0,2026-10-01,Transferencia recibida,5000.0,SI,,NaN
8,2026-10-03,FAC-1002,Cobro cliente B,3400.0,2026-10-03,Transferencia recibida,3400.0,SI,,NaN
9,2026-10-08,FAC-1003,Cobro cliente C,4500.0,2026-10-08,Transferencia recibida,4550.0,NO,,-50.0


In [17]:
# Unificamos todas las validaciones en una sola columna 'Estado_Final'
condiciones = [
    (df["Monto_x"] == df["Monto_y"]),  # Conciliado perfecto
    (df["Monto_x"].notna() & df["Monto_y"].notna() & (df["Monto_x"] != df["Monto_y"])), # Diferencia matemática
    (df["Monto_x"].notna() & df["Monto_y"].isna()), # Solo en libros
    (df["Monto_x"].isna() & df["Monto_y"].notna())  # Solo en banco
]
opciones = ["Conciliado", "Diferencia de Montos", "Partida en Tránsito", "Nota no Registrada"]

df["Estado_Final"] = np.select(condiciones, opciones, default="Revisión Manual")

# Organizamos las columnas para el reporte final (limpiamos el DataFrame)
# Renombramos sufijos x e y para que sea legible en Excel
df = df.rename(columns={
    "Fecha_x": "Fecha_Libro", "Concepto": "Concepto_Libro", "Monto_x": "Monto_Libro",
    "Fecha_y": "Fecha_Banco", "Descripcion_Banco": "Concepto_Banco", "Monto_y": "Monto_Banco"
})

In [ ]:
# Clasificamos el origen del duplicado segun en que hoja aparece repetido
df["Origen_Duplicado"] = np.select(
    [df["Duplicado_Libro"].eq("SI") & df["Duplicado_Banco"].eq("SI"),
     df["Duplicado_Libro"].eq("SI"),
     df["Duplicado_Banco"].eq("SI")],
    ["Duplicado en ambos lados", "Duplicado solo en Libro", "Duplicado solo en Banco"],
    default=""
)

# Dejamos solo las columnas relevantes
columnas_finales = ["Referencia", "Fecha_Libro", "Concepto_Libro", "Monto_Libro", 
                    "Fecha_Banco", "Concepto_Banco", "Monto_Banco", 
                    "Diferencia de Montos", "Estado_Final",
                    "Duplicado_Libro", "Duplicado_Banco", "Origen_Duplicado"]
df_reporte = df[columnas_finales].fillna("-") # Rellenamos nulos con un guion

In [29]:
df

,Fecha_Libro,Referencia,Concepto_Libro,Monto_Libro,Fecha_Banco,Concepto_Banco,Monto_Banco,Coinciliado,Partida en Tránsito / Nota no Registrada,Diferencia de Montos,Estado_Final,duplicados
0,2026-10-02,CHQ-001,Pago proveedor X,-1200.0,2026-10-04,Cobro cheque taquilla,-1200.0,SI,,NaN,Conciliado,limpio
1,2026-10-05,CHQ-002,Pago arriendo,-2000.0,NaT,NaN,NaN,NO,Partida en Tránsito,NaN,Partida en Tránsito,limpio
2,2026-10-29,CHQ-003,Pago proveedor W,-1900.0,NaT,NaN,NaN,NO,Partida en Tránsito,NaN,Partida en Tránsito,limpio
3,2026-10-21,CHQ-005,Pago proveedor Z,-800.0,2026-10-21,Cobro cheque taquilla,-800.0,SI,,NaN,Conciliado,limpio
4,2026-10-22,CHQ-005,Pago proveedor Z,-800.0,2026-10-21,Cobro cheque taquilla,-800.0,SI,,NaN,Conciliado,duplicado
5,NaT,DEV-001,NaN,NaN,2026-10-17,Nota debito cheque devuelto,-6200.0,NO,Nota no Registrada,NaN,Nota no Registrada,limpio
6,NaT,DOM-001,NaN,NaN,2026-10-27,Debito automatico seguro,-350.0,NO,Nota no Registrada,NaN,Nota no Registrada,limpio
7,2026-10-01,FAC-1001,Cobro cliente A,5000.0,2026-10-01,Transferencia recibida,5000.0,SI,,NaN,Conciliado,limpio
8,2026-10-03,FAC-1002,Cobro cliente B,3400.0,2026-10-03,Transferencia recibida,3400.0,SI,,NaN,Conciliado,limpio
9,2026-10-08,FAC-1003,Cobro cliente C,4500.0,2026-10-08,Transferencia recibida,4550.0,NO,,-50.0,Diferencia de Montos,limpio


In [ ]:
# Exportar el resultado a un nuevo archivo de Excel
nombre_archivo_salida = 'Reporte_Conciliacion_Automatizada.xlsx'

# Usamos ExcelWriter para guardar el resumen y el detalle
with pd.ExcelWriter(nombre_archivo_salida, engine='openpyxl') as writer:
    # 1. Hoja de Detalle (el DataFrame completo)
    df_reporte.to_excel(writer, sheet_name="Detalle_Conciliacion", index=False)
    
    # 2. Hoja de Resumen (Tabla dinamica generada por Pandas)
    resumen = df_reporte['Estado_Final'].value_counts().reset_index()
    resumen.columns = ['Estado', 'Cantidad de Transacciones']
    resumen.to_excel(writer, sheet_name="Resumen_Ejecutivo", index=False)

    # 3. Hoja de Referencias Sospechosas (misma normalizada, texto distinto)
    sospechosas.to_excel(writer, sheet_name="Referencias_Sospechosas", index=False)

    # 4. Hoja de Sin Contrapartida (no existen en el otro lado)
    sin_libro_export = sin_libro[["Fecha", "Referencia", "Monto"]].assign(Origen="Libro_Auxiliar")
    sin_banco_export = sin_banco[["Fecha", "Referencia", "Monto"]].assign(Origen="Extracto_Bancario")
    sin_contrapartida = pd.concat([sin_libro_export, sin_banco_export], ignore_index=True)
    sin_contrapartida.to_excel(writer, sheet_name="Sin_Contrapartida", index=False)

print(f"¡Reporte generado con éxito! Archivo guardado como: {nombre_archivo_salida}")

In [80]:
print(f"{(df["Coinciliado"] == "SI").sum()} Coinciliadas, {(df["Partida en Tránsito / Nota no Registrada"] == "Partida en Tránsito").sum()} en tránsito, {(df["Diferencia de Montos"].notna() & df["Diferencia de Montos"] != 0).sum()} diferencias")

14 Coinciliadas, 2 en tránsito, 3 diferencias
